# MC-Dropout under 30% missingness (capped RUL, matches `dropout_FIXED.ipynb` convention)

## What changed from `dropout_missing_FIXED_v2.ipynb`

That file never actually contained MC-Dropout: no `mc_dropout_predict`, no accept/abstain, just a 6-model point-prediction sweep duplicating `baseline_missing_30_FIXED.ipynb`. This notebook is the real thing -- same structure as `dropout_FIXED.ipynb` (train/val/test split, MC-Dropout at inference, `val_df`-calibrated threshold, official-test evaluation, accept/abstain sweep), just trained on the 30%-missing dataset with mask features (34 input dims: 17 raw + 17 observed/missing indicators) instead of the clean 17.

In [1]:
import pandas as pd
import numpy as np


In [2]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"

In [3]:
def create_masked_dataset(df, features):

    data = df.copy()
    mask_features = []

    for feature in features:
        mask_name = feature + "_mask"

        # 1 = observed, 0 = missing
        data[mask_name] = data[feature].notna().astype("float32")
        mask_features.append(mask_name)

    # Placeholder only -- the mask column tells the model whether 0 is a
    # real reading or a stand-in for missing data.
    data[features] = data[features].fillna(0.0).astype("float32")

    return data, mask_features

In [4]:
train_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\Missing_Train\processed_train_missing_30.csv")
test_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\Missing_Test\processed_test_missing_30.csv")

# train_missing["RUL"] is already capped at 125 -- confirmed directly against
# this file -- so it's used as-is, unlike the _uncapped sibling notebook
# which reconstructs the raw label.

train_masked, MASK_FEATURES = create_masked_dataset(train_missing, FEATURES)
test_masked, _ = create_masked_dataset(test_missing, FEATURES)

MODEL_FEATURES = FEATURES + MASK_FEATURES
print("Model features (raw + mask):", len(MODEL_FEATURES))

np.random.seed(42)

engine_ids = train_masked["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = train_masked[train_masked["Engine_ID"].isin(train_ids)].copy()
val_df = train_masked[train_masked["Engine_ID"].isin(val_ids)].copy()
test_df = train_masked[train_masked["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())
print("\nCapped RUL range (train):", train_masked["RUL"].min(), "-", train_masked["RUL"].max())

Model features (raw + mask): 34
Training engines: 70
Validation engines: 15
Testing engines: 15

Capped RUL range (train): 0 - 125


In [5]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)

In [6]:
X_train, y_train = create_sequences(train_df, MODEL_FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, MODEL_FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, MODEL_FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(12216, 30, 34)
(2720, 30, 34)
(2695, 30, 34)


BiLSTM (input width = 34: 17 raw features + 17 mask features)

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

bilstm = Sequential([

    Input(shape=(WINDOW, len(MODEL_FEATURES))),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

bilstm.summary()

C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │        50,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,017 (367.25 KB)

 Trainable params: 94,017 (367.25 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history_bilstm = bilstm.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=64,

    shuffle=False,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 3379.2397 - mae: 48.8632 - val_loss: 1749.8248 - val_mae: 37.2676 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1771.8535 - mae: 37.2775 - val_loss: 1749.0002 - val_mae: 37.0946 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1768.3885 - mae: 37.2042 - val_loss: 1748.9752 - val_mae: 37.0917 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1774.7539 - mae: 37.3003 - val_loss: 1748.7220 - val_mae: 37.0858 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1759.0726 - mae: 37.1575 - val_loss: 1521.3876 - val_mae: 35.1771 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1063.4250 - mae: 28.0481 - val_loss: 627.7110 - val_mae: 21.6096 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 785.5980 - mae: 22.9458 - 

In [9]:
def mc_dropout_predict(model, X, n_passes=100):

    predictions = []

    for _ in range(n_passes):
        pred = model(X, training=True).numpy().ravel()
        predictions.append(pred)

    predictions = np.array(predictions)

    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)

    return mean_pred, std_pred

In [10]:
mc_mean, mc_std = mc_dropout_predict(bilstm, X_test, n_passes=100)

In [11]:
print("Predictions:", mc_mean.shape)
print("Uncertainty:", mc_std.shape)

print("\nMean uncertainty:", mc_std.mean())
print("Min uncertainty :", mc_std.min())
print("Max uncertainty :", mc_std.max())

Predictions: (2695,)
Uncertainty: (2695,)

Mean uncertainty: 5.9285483
Min uncertainty : 1.09228
Max uncertainty : 9.385999


In [12]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, mc_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, mc_mean)))
print("R\u00b2  :", r2_score(y_test, mc_mean))

Held-out split:
MAE : 14.563693046569824
RMSE: 19.50833186633422
R²  : 0.7821714878082275


In [13]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)

## Calibrate the accept/abstain threshold on `val_df`

In [14]:
mc_cal_mean, mc_cal_std = mc_dropout_predict(bilstm, X_val, n_passes=100)

ACCEPT_RATE = 0.80
mc_threshold = np.quantile(mc_cal_std, ACCEPT_RATE)

print("MC uncertainty threshold:", mc_threshold)

MC uncertainty threshold: 7.4816713


## Official test set (capped, 30% missing -- literature-comparable convention)

In [15]:
RUL_CAP = 125

output_df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\RUL_FD001.csv")
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
output_df["RUL"] = output_df["RUL"].clip(upper=RUL_CAP)

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

X_test_final, test_engine_ids = create_test_sequences(test_masked, MODEL_FEATURES, WINDOW)

In [16]:
mc_test_mean, mc_test_std = mc_dropout_predict(bilstm, X_test_final, n_passes=100)

pred_df = pd.DataFrame({"Engine_ID": test_engine_ids, "Predicted_RUL": mc_test_mean, "Uncertainty": mc_test_std})
comparison = pred_df.merge(output_df, on="Engine_ID")
actual, mc_test_mean, mc_test_std = comparison["RUL"].values, comparison["Predicted_RUL"].values, comparison["Uncertainty"].values

In [17]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, mc_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, mc_test_mean)))
print("R\u00b2   :", r2_score(actual, mc_test_mean))
print("Score:", nasa_score(actual, mc_test_mean))

100% coverage:
MAE  : 11.537982940673828
RMSE : 15.299867237986087
R²   : 0.8542308211326599
Score: 467.6467294376781


## Accept / abstain sweep

In [18]:
accept = mc_test_std <= mc_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())

Accepted : 85
Abstained: 15
Coverage : 0.85


In [19]:
y_true_accept = actual[accept]
y_pred_accept = mc_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)

Accepted predictions:
MAE  : 11.768156051635742
RMSE : 15.759904441737435
R²   : 0.8490404486656189
Score: 431.05162825140854


In [20]:
for rate in [0.9, 0.8, 0.7]:
    threshold = np.quantile(mc_cal_std, rate)
    accept = mc_test_std <= threshold

    rmse = np.sqrt(mean_squared_error(actual[accept], mc_test_mean[accept]))
    score = nasa_score(actual[accept], mc_test_mean[accept])

    print(f"Acceptance={rate:.0%} | Accepted={accept.sum()} | RMSE={rmse:.3f} | Score={score:.1f}")

Acceptance=90% | Accepted=96 | RMSE=15.467 | Score=460.4
Acceptance=80% | Accepted=85 | RMSE=15.760 | Score=431.1
Acceptance=70% | Accepted=78 | RMSE=16.292 | Score=426.1


In [21]:
import os

METHOD = "mc_dropout"
SCENARIO = "missing_capped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = mc_test_mean
uncertainty = mc_test_std

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\mc_dropout__missing_capped.csv
